In [0]:
# 🟢 STEP 1: Create Data (UI + Notebook )
data = [
    (1, "Employees are entitled to 20 days of paid leave annually."),
    (2, "Work from home is allowed 3 days per week."),
    (3, "Medical insurance is provided to all employees."),
    (4, "Employees must complete 8 hours of work daily.")
]

df = spark.createDataFrame(data, ["id", "text"])
df.write.mode("overwrite").saveAsTable("workspace.default.company_docs")

In [0]:
# 🧠 Why CDF is Needed?
# Vector Search uses CDF to:
# Detect new rows
# Detect updates
# Keep index in sync automatically
%sql
ALTER TABLE workspace.default.company_docs
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
#check endpoint 
#client.list_endpoints()

In [0]:
#🟢 STEP 2: Create Vector Index (UI Practice)
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient(disable_notice=True)
client.create_delta_sync_index(
    endpoint_name="vector_end1",
    index_name="workspace.default.company_index",
    source_table_name="workspace.default.company_docs",
    pipeline_type="TRIGGERED",
    primary_key="id",
    embedding_source_column="text",
    embedding_model_endpoint_name="databricks-bge-large-en"
)

In [0]:
#🟢 STEP 3: Build Agent (Notebook)
from databricks_langchain import VectorSearchRetrieverTool, ChatDatabricks
from langchain.agents import create_agent

model = ChatDatabricks(
    endpoint="databricks-meta-llama-3-1-8b-instruct",
    max_tokens=200
)

vs_tool = VectorSearchRetrieverTool(
    index_name="workspace.default.company_index",
    name="company_search",
    description="You MUST use this tool for EVERY question.Do NOT answer from your own knowledge.Only answer using retrieved documents.",
    num_results=3,
    disable_notice=True   # 👈 ADD THIS
)

agent = create_agent(
    model=model,
    tools=[vs_tool],
    system_prompt="""
Answer clearly and concisely.
Answer ONLY about the asked topic.
Always use this tool to answer.
Ignore unrelated information.
"""""
)
#🟢 STEP 4: Test Chatbot (UI Practice 🔥)
response= agent.invoke({
    "messages": [{"role": "user", "content": "What is leave policy?"}]
})
print(response["messages"][-1].content)

In [0]:
#🟢 STEP 5: Log in MLflow (UI Learning)

import mlflow

input_example = {
    "messages": [
        {"role": "user", "content": "What is leave policy?"}
    ]
}

with mlflow.start_run():
    mlflow.langchain.log_model(
        lc_model="rag_agent.py",
        name="company_chatbot",
        input_example=input_example,
        pip_requirements="requirements.txt",
        resources=[
            {
                "type": "vector_search_index",
                "name": "workspace.default.company_index"
            },
            {
                "type": "serving_endpoint",
                "name": "databricks-meta-llama-3-1-8b-instruct"
            }
        ]
    )

In [0]:
#get run id
run_id = mlflow.active_run().info.run_id
print(run_id)

In [0]:
#🟢 STEP 6: Register Model

import mlflow

mlflow.register_model(
    model_uri="runs:/22d371b2ec4f43fd821de5ceece9e490/model",
    name="workspace.default.company_chatbot_model"
)

In [0]:
#🟢 STEP 6: Register Model (UI)